Python / ML + GenAI – Review rating regression with text embeddings (intermediate–advanced)
You have a tiny set of product reviews. Save as day5_reviews.csv:

text
review_id,user_id,review_text,stars
1,101,"Great sound quality and battery life, very happy with these headphones.",5
2,102,"Average performance, works fine but nothing special.",3
3,103,"Terrible build, broke after a week of light use.",1
4,104,"Comfortable to wear and the noise cancelling is impressive.",4
5,105,"Not worth the price, sound is weak and tinny.",2
6,106,"Excellent value for money, impressive sound for this price range.",5
7,107,"Decent for casual use, but uncomfortable after long sessions.",3
8,108,"Fantastic bass and very comfortable, exceeded expectations.",5
9,109,"Mic quality is poor and connection drops sometimes.",2
10,110,"Solid product overall, a few minor issues but good purchase.",4
Goal:

Predict stars (1–5) using both simple numeric/text features and an embedding from a Hugging Face model.

Tasks:

Load data and basic EDA:
Show head, basic stats of stars, and a count of each star rating.

Classic ML baseline:

Create a simple numeric/text feature like review_length (number of characters or tokens).
Use review_length alone in a small LinearRegression or RandomForestRegressor baseline (train/test split).
Compute RMSE and maybe MAE on the test set.

GenAI / embeddings step:
Use a Hugging Face sentence embedding model via sentence-transformers or transformers (e.g., sentence-transformers/all-MiniLM-L6-v2 or similar small model) to convert review_text into vectors.
Build an embedding matrix X_embed (one row per review).

Combined model:

Concatenate review_length with the embedding vectors (e.g., using NumPy).
Train a regression model (e.g., RandomForestRegressor or Ridge) on the combined features.
Evaluate again (RMSE, MAE) and compare with the baseline.

Interpretation:
Comment on:
Whether embeddings improved performance on this tiny dataset.
When, in a real project, you might choose classic features only vs embeddings vs a full fine‑tuned text model.
You do not need to fine‑tune any transformer—just use the pre‑trained embedding model.

In [2]:
import pandas as pd
from io import StringIO

data = """review_id,user_id,review_text,stars
1,101,"Great sound quality and battery life, very happy with these headphones.",5
2,102,"Average performance, works fine but nothing special.",3
3,103,"Terrible build, broke after a week of light use.",1
4,104,"Comfortable to wear and the noise cancelling is impressive.",4
5,105,"Not worth the price, sound is weak and tinny.",2
6,106,"Excellent value for money, impressive sound for this price range.",5
7,107,"Decent for casual use, but uncomfortable after long sessions.",3
8,108,"Fantastic bass and very comfortable, exceeded expectations.",5
9,109,"Mic quality is poor and connection drops sometimes.",2
10,110,"Solid product overall, a few minor issues but good purchase.",4"""

df = pd.read_csv(StringIO(data))

In [3]:
df.head()

,review_id,user_id,review_text,stars
0,1,101,"Great sound quality and battery life, very hap...",5
1,2,102,"Average performance, works fine but nothing sp...",3
2,3,103,"Terrible build, broke after a week of light use.",1
3,4,104,Comfortable to wear and the noise cancelling i...,4
4,5,105,"Not worth the price, sound is weak and tinny.",2


In [4]:
df.describe()

,review_id,user_id,stars
count,10.00000,10.00000,10.000000
mean,5.50000,105.50000,3.400000
std,3.02765,3.02765,1.429841
min,1.00000,101.00000,1.000000
25%,3.25000,103.25000,2.250000
50%,5.50000,105.50000,3.500000
75%,7.75000,107.75000,4.750000
max,10.00000,110.00000,5.000000


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_id    10 non-null     int64 
 1   user_id      10 non-null     int64 
 2   review_text  10 non-null     object
 3   stars        10 non-null     int64 
dtypes: int64(3), object(1)
memory usage: 452.0+ bytes


In [6]:
df['Review_length'] = df['review_text'].str.len()

X = df[['Review_length']]
y = df[['stars']]

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.3,random_state = 42)

In [8]:
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [9]:
y_pred = lr.predict(X_test)
y_pred

array([[2.47881088],
       [2.62333966],
       [4.50221379]])

In [10]:
y_test

,stars
8,2
1,3
5,5


In [25]:
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score

RMSE = round(root_mean_squared_error(y_test, y_pred),2)
print("RMSE without embeddings",RMSE)

r2 = round(r2_score(y_test,y_pred),2)
print("r2 Score without embeddings",r2)

RMSE without embeddings 0.45
r2 Score without embeddings 0.87


In [12]:
# !pip install sentence-transformers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

X_embed = model.encode(df['review_text'].tolist(), convert_to_numpy= True)

In [14]:
print(X_embed)

[[-0.08975598  0.02633685 -0.03256745 ... -0.01721612  0.01549706
   0.08387356]
 [ 0.01252003 -0.02533148 -0.02988353 ... -0.06437899 -0.0429428
   0.05456759]
 [-0.03394102  0.06857027  0.05841081 ... -0.09691541 -0.02398281
   0.16571754]
 ...
 [-0.06799167 -0.0075865   0.00328843 ... -0.02780241 -0.02934998
   0.02486188]
 [ 0.00203607 -0.0501302   0.04312417 ...  0.00711528 -0.02542583
   0.02077935]
 [-0.09646957  0.01574835 -0.00756191 ... -0.01567675  0.00711208
   0.10361154]]


In [13]:
print(X_embed.shape)

(10, 384)


In [15]:
import numpy as np
df['review_length'] = df['review_text'].astype(str).str.len()


In [16]:
df['review_length']

0    71
1    52
2    48
3    59
4    45
5    65
6    61
7    59
8    51
9    60
Name: review_length, dtype: int64

In [17]:
import numpy as np
review_length = df['review_length'].values.reshape(-1,1)
X_combined = np.hstack((review_length, X_embed))
print("Combined feature matrix shape:", X_combined.shape)

Combined feature matrix shape: (10, 385)


In [20]:
from sklearn.model_selection import train_test_split
y = df['stars']  # target labels
X_train, X_test, y_train, y_test = train_test_split( X_combined, y, test_size=0.3, random_state=42)

In [21]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge

rf = RandomForestRegressor(n_estimators = 100, random_state = 42)
rf.fit(X_train, y_train)

ridge = Ridge(alpha = 1.0)
ridge.fit(X_train, y_train)

Ridge()

In [26]:
y_pred_rf = rf.predict(X_test)
y_pred_ridge = ridge.predict(X_test)

from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error

print("After Embeddings")
print("RandomForest RMSE {}".format(root_mean_squared_error(y_test,y_pred_rf)))
print("RandomForest r2 {}".format(r2_score(y_test,y_pred_rf)))
print("RandomForest MAE {}".format(mean_absolute_error(y_test,y_pred_rf)))

print("\n")

print("Ridge RMSE {}".format(root_mean_squared_error(y_test,y_pred_ridge)))
print("Ridge r2 {}".format(r2_score(y_test,y_pred_ridge)))
print("RandomForest MAE {}".format(mean_absolute_error(y_test,y_pred_ridge)))

After Embeddings
RandomForest RMSE 1.1590513362228612
RandomForest r2 0.13638571428571422
RandomForest MAE 1.0333333333333334


Ridge RMSE 0.41411298972908467
Ridge r2 0.8897567061170536
RandomForest MAE 0.4080713965209881


Key Takeaways
1. Embeddings didn’t help much here because your dataset is too small. With only 10 reviews, the model can’t learn meaningful patterns from 384 embedding dimensions.

2. Classic features (like review_length) can outperform embeddings on tiny datasets because they’re simple, low‑dimensional, and less prone to overfitting.

In real projects:

1. If you have small data → stick to classic features (length, word counts, sentiment scores).

2. If you have moderate data (hundreds/thousands of samples) → embeddings start to shine, especially with regularized models like Ridge or linear regression.

3. If you have large data (tens of thousands+) → embeddings + fine‑tuned text models (like BERT or GPT‑based regressors/classifiers) usually outperform classic features.